# Teste de Extração: PPM (Pesquisa Pecuária Municipal)

Extrai o efetivo de rebanhos via API SIDRA (Tabela 3939).

## URL da API SIDRA (março/2026)

| Tabela | Classificação | Descrição | URL Padrão |
|--------|---------------|-----------|------------|
| **3939** | c79/{COD} | Efetivo de Rebanhos | `https://apisidra.ibge.gov.br/values/t/3939/n6/all/v/all/p/{ANO}/c79/{COD}` |

## Tipos de Rebanho Disponíveis

| Código | Rebanho | Código | Rebanho |
|--------|---------|--------|----------|
| 2670 | Bovinos | 2731 | Caprinos |
| 2685 | Bubalinos | 2740 | Suínos (total) |
| 2693 | Equinos | 2758 | Suínos (matrizes) |
| 2707 | Asininos | 2766 | Galináceos (total) |
| 2715 | Muar | 2774 | Galinhas |
| 2723 | Ovinos | 2782 | Codornas |

## Status dos Dados

- **2021-2023:** Dados consolidados
- **2024:** Disponível desde setembro/2025
- **Nível territorial:** n6 = Municípios

## Troubleshooting

| Problema | Solução |
|----------|----------|
| Erro 503 | Retry com backoff + máximo 3 workers |
| sidrapy instável | Usar requests direto |
| Dados vazios | Ano/código sem dados disponíveis |
| Header como dado | Remover linha com `iloc[1:]` |

In [ ]:
import requests
import pandas as pd
import uuid
import time
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, Dict, List

# ====================== CONFIG ======================
DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento de códigos → nomes amigáveis
REBANHOS = {
    "2670": "bovinos",
    "2685": "bubalinos",
    "2693": "equinos",
    "2707": "asininos",
    "2715": "muar",
    "2723": "ovinos",
    "2731": "caprinos",
    "2740": "suinos_total",
    "2758": "suinos_matrizes",
    "2766": "galinaceos_total",
    "2774": "galinhas",
    "2782": "codornas",
}

def salvar_parquet(df: pd.DataFrame, dataset: str, ano: int):
    """Salva DataFrame em Parquet particionado por ano."""
    path = BRONZE_DIR / dataset / f"ano={ano}"
    path.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path / f"chunk_{uuid.uuid4().hex[:8]}.parquet", index=False)

# ====================== FUNÇÃO DE DOWNLOAD UNITÁRIA ======================
def baixar_um_rebanho(
    ano: int,
    cod_rebanho: str,
    nome_rebanho: str,
    session: requests.Session
) -> tuple[Optional[pd.DataFrame], str, int]:
    """Baixa dados de um rebanho específico com retry automático."""
    url = f"https://apisidra.ibge.gov.br/values/t/3939/n6/all/v/all/p/{ano}/c79/{cod_rebanho}"
    
    for tentativa in range(5):
        try:
            r = session.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()  # ← API retorna JSON
            
            if not data or not isinstance(data, list) or len(data) <= 1:
                return None, nome_rebanho, ano
            
            df = pd.DataFrame(data)
            # Remove header se vier (raro)
            if 'D1C' in df.columns and str(df['D1C'].iloc[0]).lower() == 'ano':
                df = df.iloc[1:].copy()
            
            return df, nome_rebanho, ano
            
        except Exception as e:
            print(f"  Tentativa {tentativa+1}/5 - {nome_rebanho} {ano}: {e}")
            time.sleep(2 ** tentativa)  # backoff: 1s, 2s, 4s, 8s, 16s
    
    print(f"  ❌ Falha final: {nome_rebanho} {ano}")
    return None, nome_rebanho, ano

# ====================== EXTRATOR COM PARALELISMO ======================
class ExtratorPPMRebanhos:
    def __init__(self, max_workers: int = 3):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) PPM-Rebanhos-Downloader/2026'
        })
        self.max_workers = max_workers

    def extrair_e_salvar(self, anos: List[int], codigos_selecionados: List[str] = None):
        """Extrai e salva dados de múltiplos rebanhos em paralelo."""
        if codigos_selecionados is None:
            codigos_selecionados = list(REBANHOS.keys())
        
        tarefas = []
        for ano in anos:
            for cod in codigos_selecionados:
                nome = REBANHOS[cod]
                tarefas.append((ano, cod, nome))
        
        total_tarefas = len(tarefas)
        print(f"Iniciando {total_tarefas} downloads ({len(anos)} anos × {len(codigos_selecionados)} tipos de rebanho)")
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_info = {
                executor.submit(
                    baixar_um_rebanho, ano, cod, nome, self.session
                ): (ano, nome)
                for ano, cod, nome in tarefas
            }
            
            with tqdm(total=total_tarefas, desc="PPM Rebanhos", unit="download") as pbar:
                for future in as_completed(future_to_info):
                    df, nome_rebanho, ano = future.result()
                    if df is not None and not df.empty:
                        dataset = f"ppm_{nome_rebanho}"
                        salvar_parquet(df, dataset, ano)
                        print(f"  ✅ Salvo: ppm_{nome_rebanho}/ano={ano} ({len(df):,} linhas)")
                    else:
                        print(f"  -- Sem dados ou falha: {nome_rebanho} {ano}")
                    
                    pbar.update(1)
                    time.sleep(0.3)  # delay extra para ajudar rate limit

# ====================== EXECUÇÃO ======================
if __name__ == "__main__":
    anos = [2021, 2022, 2023, 2024]  # 2024 disponível desde set/2025
    
    # Para testar com menos tipos (exemplo):
    # codigos_selecionados = ["2670", "2723", "2731", "2740"]  # bovinos, ovinos, caprinos, suínos_total
    
    codigos_selecionados = None  # None = todos
    
    extrator = ExtratorPPMRebanhos(max_workers=3)  # ajuste para 2 se der muitos 503
    extrator.extrair_e_salvar(anos, codigos_selecionados)
    
    print("\nFinalizado! Verifique as pastas em data/bronze/ppm_*")